# Anexo II. Código de la aplicación de valoración automatizada

Códigos A5.1–A5.20. El visor utiliza CartoDB Positron como mapa base para la representación cartográfica.

## Preparación del entorno

In [ ]:
!pip install geopandas folium streamlit streamlit-folium xgboost joblib


## A5.1. Entrenamiento y exportación del modelo XGBoost B

In [ ]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

df = pd.read_csv("dataset_modelo_hortaleza.csv")
y = df["price"]

variables_predictoras = [
    "size", "rooms", "bathrooms", "floor", "hasLift", "status",
    "propertyType", "typology", "exterior", "newDevelopment",
    "latitude", "longitude", "dist_metro", "dist_bus", "dist_school",
    "dist_hospital", "dist_health_center", "dist_supermarket",
    "dist_park", "dist_sports"
]

X = df[variables_predictoras]

variables_categoricas = [
    "floor", "hasLift", "status", "propertyType", "typology", "exterior"
]

variables_numericas = [
    v for v in variables_predictoras if v not in variables_categoricas
]

preprocesamiento = ColumnTransformer(
    transformers=[
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         variables_categoricas),
        ("numericas", "passthrough", variables_numericas)
    ]
)

modelo_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

modelo_xgb_b = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", modelo_xgb)
])

modelo_xgb_b.fit(X, y)

joblib.dump(modelo_xgb_b, "modelo_xgb_b_hortaleza.joblib")

print("Modelo XGBoost B entrenado y exportado correctamente.")


## A5.2. Carga y comprobación del modelo

In [ ]:
modelo = joblib.load("modelo_xgb_b_hortaleza.joblib")

print("Tipo de modelo:")
print(type(modelo))

print("\nComponentes del pipeline:")
print(modelo.named_steps.keys())


## A5.3. Carga de las capas geográficas

In [4]:
import geopandas as gpd

archivo_gpkg = "visor_hortaleza.gpkg"

limite = gpd.read_file(archivo_gpkg, layer="limite_hortaleza").to_crs("EPSG:4326")
viviendas = gpd.read_file(archivo_gpkg, layer="viviendas").to_crs("EPSG:4326")
metro = gpd.read_file(archivo_gpkg, layer="metro").to_crs("EPSG:4326")
bus = gpd.read_file(archivo_gpkg, layer="bus").to_crs("EPSG:4326")
school = gpd.read_file(archivo_gpkg, layer="school").to_crs("EPSG:4326")
hospital = gpd.read_file(archivo_gpkg, layer="hospital").to_crs("EPSG:4326")
health_center = gpd.read_file(archivo_gpkg, layer="health_center").to_crs("EPSG:4326")
supermarket = gpd.read_file(archivo_gpkg, layer="supermarket").to_crs("EPSG:4326")
park = gpd.read_file(archivo_gpkg, layer="park").to_crs("EPSG:4326")
sports = gpd.read_file(archivo_gpkg, layer="sports").to_crs("EPSG:4326")


## A5.4. Preparación de las capas para el cálculo de distancias

In [ ]:
capas_visores = {
    "metro": metro.to_crs("EPSG:25830"),
    "bus": bus.to_crs("EPSG:25830"),
    "school": school.to_crs("EPSG:25830"),
    "hospital": hospital.to_crs("EPSG:25830"),
    "health_center": health_center.to_crs("EPSG:25830"),
    "supermarket": supermarket.to_crs("EPSG:25830"),
    "park": park.to_crs("EPSG:25830"),
    "sports": sports.to_crs("EPSG:25830")
}

for nombre, capa in capas_visores.items():
    print(f"{nombre:<18} {len(capa):>5} elementos | {capa.crs}")


## A5.5. Comprobación de capas, dimensiones y CRS

In [ ]:
capas = [
    "limite_hortaleza", "viviendas", "metro", "bus", "school",
    "hospital", "health_center", "supermarket", "park", "sports"
]

for capa in capas:
    gdf = gpd.read_file("visor_hortaleza.gpkg", layer=capa)
    print(f"{capa:<18} Dimensiones: {gdf.shape} | CRS: {gdf.crs}")


## A5.6. Creación del mapa base

In [ ]:
import folium

centro = limite.geometry.iloc[0].centroid

mapa = folium.Map(
    location=[centro.y, centro.x],
    zoom_start=13,
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    control_scale=True
)

folium.GeoJson(
    limite,
    name="Distrito de Hortaleza"
).add_to(mapa)

mapa

## A5.7. Representación interactiva de las viviendas

In [ ]:
grupo_viviendas = folium.FeatureGroup(
    name="Viviendas del dataset",
    show=True
)

for _, vivienda in viviendas.iterrows():
    folium.CircleMarker(
        location=[vivienda.geometry.y, vivienda.geometry.x],
        radius=4,
        fill=True,
        fill_opacity=0.75,
        weight=1,
        tooltip="Vivienda del dataset"
    ).add_to(grupo_viviendas)

grupo_viviendas.add_to(mapa)
mapa


## A5.8. Incorporación de las capas de elementos de interés

In [ ]:
def añadir_capa_iconos(mapa, datos, nombre, icono, color, mostrar=False):
    grupo = folium.FeatureGroup(name=nombre, show=mostrar)

    for _, elemento in datos.iterrows():
        folium.Marker(
            location=[elemento.geometry.y, elemento.geometry.x],
            icon=folium.Icon(icon=icono, prefix="fa", color=color),
            tooltip=nombre
        ).add_to(grupo)

    grupo.add_to(mapa)

añadir_capa_iconos(mapa, metro, "Metro", "subway", "red")
añadir_capa_iconos(mapa, bus, "Paradas de autobús", "bus", "blue")
añadir_capa_iconos(mapa, school, "Colegios", "graduation-cap", "green")
añadir_capa_iconos(mapa, hospital, "Hospitales", "hospital-o", "darkred")
añadir_capa_iconos(mapa, health_center, "Centros de salud", "medkit", "orange")
añadir_capa_iconos(mapa, supermarket, "Supermercados", "shopping-cart", "purple")
añadir_capa_iconos(mapa, park, "Parques", "tree", "green")
añadir_capa_iconos(mapa, sports, "Instalaciones deportivas", "futbol-o", "black")

mapa


## A5.9. Control de capas

In [ ]:
folium.LayerControl(collapsed=False).add_to(mapa)
mapa


## A5.10. Captura de la ubicación mediante clic en el mapa

In [ ]:
from streamlit_folium import st_folium

resultado_mapa = st_folium(
    mapa,
    width=None,
    height=600
)

if resultado_mapa["last_clicked"] is not None:
    latitud = resultado_mapa["last_clicked"]["lat"]
    longitud = resultado_mapa["last_clicked"]["lng"]


## A5.11. Cálculo de las ocho distancias geoespaciales

In [12]:
from shapely.geometry import Point

def calcular_distancias_nueva_vivienda(latitud, longitud, capas_visores):
    punto_wgs84 = gpd.GeoDataFrame(
        geometry=[Point(longitud, latitud)],
        crs="EPSG:4326"
    )

    punto_utm = punto_wgs84.to_crs(
        "EPSG:25830"
    ).geometry.iloc[0]

    nombres_variables = {
        "metro": "dist_metro",
        "bus": "dist_bus",
        "school": "dist_school",
        "hospital": "dist_hospital",
        "health_center": "dist_health_center",
        "supermarket": "dist_supermarket",
        "park": "dist_park",
        "sports": "dist_sports"
    }

    distancias = {}

    for nombre_capa, nombre_variable in nombres_variables.items():
        capa = capas_visores[nombre_capa]
        distancias[nombre_variable] = float(
            capa.geometry.distance(punto_utm).min()
        )

    return distancias


## A5.12. Construcción de las 20 variables predictoras

In [ ]:
# Ejemplo de características introducidas por el usuario.
size = 100
rooms = 3
bathrooms = 2
floor = "3"
hasLift = "True"
status = "good"
propertyType = "flat"
typology = "flat"
exterior = "True"
newDevelopment = 0

# Para la comprobación del flujo en el notebook se utiliza
# la ubicación de una vivienda existente del dataset.
latitud = float(viviendas.geometry.iloc[0].y)
longitud = float(viviendas.geometry.iloc[0].x)

distancias = calcular_distancias_nueva_vivienda(
    latitud,
    longitud,
    capas_visores
)

variables_modelo = [
    "size", "rooms", "bathrooms", "floor", "hasLift",
    "status", "propertyType", "typology", "exterior",
    "newDevelopment", "latitude", "longitude", "dist_metro",
    "dist_bus", "dist_school", "dist_hospital",
    "dist_health_center", "dist_supermarket",
    "dist_park", "dist_sports"
]

X_nueva = pd.DataFrame([{
    "size": size,
    "rooms": rooms,
    "bathrooms": bathrooms,
    "floor": floor,
    "hasLift": hasLift,
    "status": status,
    "propertyType": propertyType,
    "typology": typology,
    "exterior": exterior,
    "newDevelopment": newDevelopment,
    "latitude": latitud,
    "longitude": longitud,
    **distancias
}])

X_nueva = X_nueva[variables_modelo]

print("Número de variables predictoras:", X_nueva.shape[1])
X_nueva


## A5.13. Predicción del valor del inmueble

In [ ]:
precio_estimado = modelo.predict(X_nueva)[0]

print(f"Valor estimado: {precio_estimado:,.0f} €")


## A5.14. Función reutilizable de valoración

In [15]:
def valorar_nueva_vivienda(
    modelo, latitud, longitud, size, rooms, bathrooms,
    floor, hasLift, status, propertyType, typology,
    exterior, newDevelopment, capas_visores
):
    distancias = calcular_distancias_nueva_vivienda(
        latitud, longitud, capas_visores
    )

    X_nueva = pd.DataFrame([{
        "size": size,
        "rooms": rooms,
        "bathrooms": bathrooms,
        "floor": floor,
        "hasLift": hasLift,
        "status": status,
        "propertyType": propertyType,
        "typology": typology,
        "exterior": exterior,
        "newDevelopment": newDevelopment,
        "latitude": latitud,
        "longitude": longitud,
        **distancias
    }])

    X_nueva = X_nueva[variables_modelo]
    precio = modelo.predict(X_nueva)[0]

    return float(precio), X_nueva


## A5.15. Configuración de la aplicación Streamlit

In [ ]:
import streamlit as st

st.set_page_config(
    page_title="AVM Hortaleza",
    page_icon="🏠",
    layout="wide"
)

st.title("Valoración automatizada de inmuebles")
st.subheader("Caso de estudio: distrito de Hortaleza (Madrid)")


## A5.16. Formulario de características del inmueble

In [ ]:
with st.form("formulario_inmueble"):

    size = st.number_input(
        "Superficie (m²)",
        min_value=20.0,
        max_value=2000.0,
        value=100.0,
        step=1.0
    )

    rooms = st.number_input(
        "Habitaciones",
        min_value=0,
        max_value=15,
        value=3,
        step=1
    )

    bathrooms = st.number_input(
        "Baños",
        min_value=0,
        max_value=15,
        value=2,
        step=1
    )

    floor = st.selectbox(
        "Planta",
        [
            "bj", "en", "st", "1", "2", "3", "4", "5", "6",
            "7", "8", "9", "10", "14", "15", "16", "17",
            "18", "19", "20", "21", "unknown"
        ],
        index=6
    )

    hasLift = st.selectbox(
        "Ascensor",
        ["False", "True", "unknown"],
        index=1
    )

    status = st.selectbox(
        "Estado",
        ["good", "newdevelopment", "renew", "unknown"],
        index=0
    )

    propertyType = st.selectbox(
        "Tipo de inmueble",
        ["chalet", "duplex", "flat", "penthouse", "studio"],
        index=2
    )

    typology = st.selectbox(
        "Tipología",
        ["chalet", "flat"],
        index=1
    )

    exterior = st.selectbox(
        "Exterior",
        ["False", "True", "unknown"],
        index=1
    )

    newDevelopment = st.selectbox(
        "Obra nueva",
        [0, 1],
        index=0
    )

    valorar = st.form_submit_button("🔎 Valorar inmueble")


## A5.17. Información emergente de las viviendas

In [18]:
grupo_viviendas = folium.FeatureGroup(
    name="Viviendas del dataset",
    show=True
)

for _, vivienda in viviendas.iterrows():

    superficie = vivienda.get("size", "N/D")
    habitaciones = vivienda.get("rooms", "N/D")
    banos = vivienda.get("bathrooms", "N/D")
    tipo = vivienda.get("propertyType", "N/D")
    precio = vivienda.get("price", "N/D")

    if pd.notna(precio):
        precio_texto = f"{precio:,.0f} €"
    else:
        precio_texto = "N/D"

    popup_html = (
        "<div style='font-family: Arial; width: 230px; line-height: 1.6;'>"
        "<h4>🏠 Vivienda del dataset</h4>"
        f"<b>Superficie:</b> {superficie} m²<br>"
        f"<b>Habitaciones:</b> {habitaciones}<br>"
        f"<b>Baños:</b> {banos}<br>"
        f"<b>Tipo:</b> {tipo}<br>"
        f"<b>Precio:</b> {precio_texto}"
        "</div>"
    )

    folium.CircleMarker(
        location=[vivienda.geometry.y, vivienda.geometry.x],
        radius=4,
        fill=True,
        fill_opacity=0.75,
        weight=1,
        tooltip="🏠 Ver información",
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(grupo_viviendas)

grupo_viviendas.add_to(mapa)


## A5.18. Marcador de la nueva ubicación seleccionada

In [ ]:
if (
    "latitud_seleccionada" in st.session_state
    and "longitud_seleccionada" in st.session_state
):

    if (
        st.session_state.latitud_seleccionada is not None
        and st.session_state.longitud_seleccionada is not None
    ):

        folium.Marker(
            location=[
                st.session_state.latitud_seleccionada,
                st.session_state.longitud_seleccionada
            ],
            icon=folium.Icon(
                icon="home",
                prefix="fa",
                color="green"
            ),
            tooltip="📍 Inmueble a valorar"
        ).add_to(mapa)


## A5.19. Carga mediante caché del modelo y de los datos geográficos

In [ ]:
@st.cache_resource
def cargar_modelo():
    return joblib.load("modelo_xgb_b_hortaleza.joblib")


@st.cache_data
def cargar_datos_geograficos():

    archivo_gpkg = "visor_hortaleza.gpkg"

    limite = gpd.read_file(
        archivo_gpkg,
        layer="limite_hortaleza"
    ).to_crs("EPSG:4326")

    viviendas = gpd.read_file(
        archivo_gpkg,
        layer="viviendas"
    ).to_crs("EPSG:4326")

    nombres_capas = [
        "metro", "bus", "school", "hospital",
        "health_center", "supermarket", "park", "sports"
    ]

    capas_visores = {}

    for nombre in nombres_capas:

        capa = gpd.read_file(
            archivo_gpkg,
            layer=nombre
        ).to_crs("EPSG:25830")

        capas_visores[nombre] = capa

    return limite, viviendas, capas_visores


modelo = cargar_modelo()

limite, viviendas, capas_visores = cargar_datos_geograficos()


## A5.20. Flujo final de valoración y presentación del resultado

In [21]:
if resultado_mapa["last_clicked"] is not None:

    latitud = resultado_mapa["last_clicked"]["lat"]
    longitud = resultado_mapa["last_clicked"]["lng"]

    distancias = calcular_distancias_nueva_vivienda(
        latitud,
        longitud,
        capas_visores
    )

    X_nueva = pd.DataFrame([{
        "size": size,
        "rooms": rooms,
        "bathrooms": bathrooms,
        "floor": floor,
        "hasLift": hasLift,
        "status": status,
        "propertyType": propertyType,
        "typology": typology,
        "exterior": exterior,
        "newDevelopment": newDevelopment,
        "latitude": latitud,
        "longitude": longitud,
        **distancias
    }])

    X_nueva = X_nueva[variables_modelo]

    precio_estimado = modelo.predict(X_nueva)[0]
    precio_m2 = precio_estimado / size

    st.success(
        f"Valor estimado: {precio_estimado:,.0f} €"
    )

    st.metric(
        "Valor estimado por m²",
        f"{precio_m2:,.0f} €/m²"
    )

    st.write("### Características utilizadas")

    st.dataframe(
        X_nueva[[
            "size", "rooms", "bathrooms", "floor",
            "hasLift", "status", "propertyType",
            "typology", "exterior", "newDevelopment"
        ]]
    )


A5.21. Inicialización de la aplicación y gestión de la ubicación

In [ ]:
import streamlit as st

# Configuración de la aplicación
st.set_page_config(
    page_title="AVM Hortaleza",
    page_icon="🏠",
    layout="wide"
)

# Inicialización de las variables de sesión
if "latitud_seleccionada" not in st.session_state:
    st.session_state.latitud_seleccionada = None

if "longitud_seleccionada" not in st.session_state:
    st.session_state.longitud_seleccionada = None

print("Aplicación inicializada correctamente.")
print("Latitud seleccionada:", st.session_state.latitud_seleccionada)
print("Longitud seleccionada:", st.session_state.longitud_seleccionada)

A5.22. Carga del modelo predictivo

In [ ]:
import joblib

RUTA_MODELO = "modelo_xgb_b_hortaleza.joblib"

def cargar_modelo():
    modelo = joblib.load(RUTA_MODELO)
    return modelo

modelo = cargar_modelo()

print("Modelo cargado correctamente.")
print("Tipo de modelo:", type(modelo))

A5.23. Carga de las capas geoespaciales

In [ ]:
import geopandas as gpd

archivo_gpkg = "visor_hortaleza.gpkg"

limite = gpd.read_file(
    archivo_gpkg,
    layer="limite_hortaleza"
).to_crs("EPSG:4326")

viviendas = gpd.read_file(
    archivo_gpkg,
    layer="viviendas"
).to_crs("EPSG:4326")

metro = gpd.read_file(
    archivo_gpkg,
    layer="metro"
).to_crs("EPSG:4326")

bus = gpd.read_file(
    archivo_gpkg,
    layer="bus"
).to_crs("EPSG:4326")

school = gpd.read_file(
    archivo_gpkg,
    layer="school"
).to_crs("EPSG:4326")

hospital = gpd.read_file(
    archivo_gpkg,
    layer="hospital"
).to_crs("EPSG:4326")

health_center = gpd.read_file(
    archivo_gpkg,
    layer="health_center"
).to_crs("EPSG:4326")

supermarket = gpd.read_file(
    archivo_gpkg,
    layer="supermarket"
).to_crs("EPSG:4326")

park = gpd.read_file(
    archivo_gpkg,
    layer="park"
).to_crs("EPSG:4326")

sports = gpd.read_file(
    archivo_gpkg,
    layer="sports"
).to_crs("EPSG:4326")

print("Capas geográficas cargadas correctamente.")
print("Límite:", len(limite))
print("Viviendas:", len(viviendas))
print("Metro:", len(metro))
print("Bus:", len(bus))
print("Colegios:", len(school))
print("Hospitales:", len(hospital))
print("Centros de salud:", len(health_center))
print("Supermercados:", len(supermarket))
print("Parques:", len(park))
print("Instalaciones deportivas:", len(sports))

A5.24. Preparación de las capas para el cálculo de distancias

In [ ]:
capas_visores = {
    "metro": metro.to_crs("EPSG:25830"),
    "bus": bus.to_crs("EPSG:25830"),
    "school": school.to_crs("EPSG:25830"),
    "hospital": hospital.to_crs("EPSG:25830"),
    "health_center": health_center.to_crs("EPSG:25830"),
    "supermarket": supermarket.to_crs("EPSG:25830"),
    "park": park.to_crs("EPSG:25830"),
    "sports": sports.to_crs("EPSG:25830")
}

print("Capas preparadas para el cálculo de distancias.")
print()

for nombre, capa in capas_visores.items():
    print(
        f"{nombre}: "
        f"{len(capa)} elementos | "
        f"CRS: {capa.crs}"
    )

A5.25. Definición de la función de cálculo de distancias

In [ ]:
import numpy as np
from shapely.geometry import Point

def calcular_distancias_nueva_vivienda(
    latitud,
    longitud,
    capas_visores
):
    punto = gpd.GeoDataFrame(
        geometry=[Point(longitud, latitud)],
        crs="EPSG:4326"
    ).to_crs("EPSG:25830")

    distancias = {}

    correspondencias = {
        "metro": "dist_metro",
        "bus": "dist_bus",
        "school": "dist_school",
        "hospital": "dist_hospital",
        "health_center": "dist_health_center",
        "supermarket": "dist_supermarket",
        "park": "dist_park",
        "sports": "dist_sports"
    }

    for nombre_capa, nombre_variable in correspondencias.items():
        capa = capas_visores[nombre_capa]

        if capa.empty:
            distancias[nombre_variable] = np.nan
        else:
            distancias[nombre_variable] = (
                punto.geometry.iloc[0]
                .distance(capa.geometry)
                .min()
            )

    return distancias


print("Función de cálculo de distancias definida correctamente.")

A5.26. Prueba del cálculo de distancias

In [ ]:
# Coordenadas de prueba dentro del distrito de Hortaleza
latitud_prueba = 40.4700
longitud_prueba = -3.6400

distancias_prueba = calcular_distancias_nueva_vivienda(
    latitud_prueba,
    longitud_prueba,
    capas_visores
)

print("Distancias calculadas:")
print()

for variable, distancia in distancias_prueba.items():
    print(f"{variable}: {distancia:.2f} m")

A5.27. Construcción de las 20 variables predictoras

In [ ]:
import pandas as pd

# Características de una vivienda de prueba
size = 100.0
rooms = 3
bathrooms = 2
floor = "4"
hasLift = "True"
status = "good"
propertyType = "flat"
typology = "flat"
exterior = "True"
newDevelopment = 0

# Construcción de las 20 variables predictoras
X_nueva = pd.DataFrame(
    [{
        "size": size,
        "rooms": rooms,
        "bathrooms": bathrooms,
        "floor": floor,
        "hasLift": hasLift,
        "status": status,
        "propertyType": propertyType,
        "typology": typology,
        "exterior": exterior,
        "newDevelopment": newDevelopment,
        "latitude": latitud_prueba,
        "longitude": longitud_prueba,
        "dist_metro": distancias_prueba["dist_metro"],
        "dist_bus": distancias_prueba["dist_bus"],
        "dist_school": distancias_prueba["dist_school"],
        "dist_hospital": distancias_prueba["dist_hospital"],
        "dist_health_center": distancias_prueba["dist_health_center"],
        "dist_supermarket": distancias_prueba["dist_supermarket"],
        "dist_park": distancias_prueba["dist_park"],
        "dist_sports": distancias_prueba["dist_sports"]
    }]
)

print("Variables predictoras construidas correctamente.")
print("Número de variables:", X_nueva.shape[1])
print()
print(X_nueva)

A5.28. Predicción de una nueva vivienda

In [ ]:
precio_estimado = modelo.predict(
    X_nueva
)[0]

print(
    f"Valor estimado: {precio_estimado:,.0f} €".replace(",", ".")
)

A5.29. Función de valoración reutilizable

In [ ]:
def valorar_nueva_vivienda(
    modelo,
    latitud,
    longitud,
    size,
    rooms,
    bathrooms,
    floor,
    hasLift,
    status,
    propertyType,
    typology,
    exterior,
    newDevelopment,
    capas_visores
):
    # Cálculo de las distancias geoespaciales
    distancias = calcular_distancias_nueva_vivienda(
        latitud,
        longitud,
        capas_visores
    )

    # Construcción de las 20 variables predictoras
    X_nueva = pd.DataFrame([{
        "size": size,
        "rooms": rooms,
        "bathrooms": bathrooms,
        "floor": floor,
        "hasLift": hasLift,
        "status": status,
        "propertyType": propertyType,
        "typology": typology,
        "exterior": exterior,
        "newDevelopment": newDevelopment,
        "latitude": latitud,
        "longitude": longitud,
        **distancias
    }])

    # Predicción del valor
    precio = modelo.predict(X_nueva)[0]

    return precio, X_nueva


print("Función de valoración definida correctamente.")

A5.30. Prueba de la función de valoración

In [34]:
precio_prueba, X_prueba = valorar_nueva_vivienda(
    modelo=modelo,
    latitud=latitud_prueba,
    longitud=longitud_prueba,
    size=100.0,
    rooms=3,
    bathrooms=2,
    floor="4",
    hasLift="True",
    status="good",
    propertyType="flat",
    typology="flat",
    exterior="True",
    newDevelopment=0,
    capas_visores=capas_visores
)

print(
    f"Valor estimado: {precio_prueba:,.0f} €".replace(",", ".")
)

print("Número de variables utilizadas:", X_prueba.shape[1])

Valor estimado: 494.517 €
Número de variables utilizadas: 20


A5.31. Configuración básica de Streamlit

In [ ]:
import streamlit as st

st.title(
    "Valoración automatizada de inmuebles"
)

st.subheader(
    "Caso de estudio: distrito de Hortaleza (Madrid)"
)

print("Interfaz de Streamlit configurada correctamente.")

A5.32. Formulario de características del inmueble

In [ ]:
with st.form("formulario_inmueble"):

    size = st.number_input(
        "Superficie (m²)",
        min_value=20.0,
        max_value=2000.0,
        value=100.0,
        step=1.0
    )

    rooms = st.number_input(
        "Número de habitaciones",
        min_value=0,
        max_value=15,
        value=3,
        step=1
    )

    bathrooms = st.number_input(
        "Número de baños",
        min_value=0,
        max_value=15,
        value=2,
        step=1
    )

    floor = st.selectbox(
        "Planta",
        [
            "bj", "en", "st", "1", "2", "3", "4", "5", "6",
            "7", "8", "9", "10", "14", "15", "16", "17",
            "18", "19", "20", "21", "unknown"
        ],
        index=6
    )

    hasLift = st.selectbox(
        "Ascensor",
        ["False", "True", "unknown"],
        index=1
    )

    status = st.selectbox(
        "Estado",
        ["good", "newdevelopment", "renew", "unknown"],
        index=0
    )

    propertyType = st.selectbox(
        "Tipo de inmueble",
        ["chalet", "duplex", "flat", "penthouse", "studio"],
        index=2
    )

    typology = st.selectbox(
        "Tipología",
        ["chalet", "flat"],
        index=1
    )

    exterior = st.selectbox(
        "Exterior",
        ["False", "True", "unknown"],
        index=1
    )

    newDevelopment = st.selectbox(
        "Obra nueva",
        [0, 1],
        index=0
    )

    boton_valorar = st.form_submit_button(
        "🔎 Valorar inmueble"
    )

print("Formulario de características creado correctamente.")

A5.33. Creación del mapa interactivo

In [ ]:
import folium

# Centro del distrito
centro = limite.geometry.iloc[0].centroid

# Creación del mapa
mapa = folium.Map(
    location=[centro.y, centro.x],
    zoom_start=13,
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    control_scale=True
)

# Añadir el límite del distrito
folium.GeoJson(
    limite,
    name="Distrito de Hortaleza"
).add_to(mapa)

print("Mapa interactivo creado correctamente.")

A5.34. Incorporación de las viviendas al mapa

In [ ]:
# Grupo de viviendas
grupo_viviendas = folium.FeatureGroup(
    name="Viviendas del dataset"
)

for _, vivienda in viviendas.iterrows():

    superficie = vivienda["size"]
    habitaciones = vivienda["rooms"]
    banos = vivienda["bathrooms"]
    tipo = vivienda["propertyType"]
    precio = vivienda["price"]

    # Formateo del precio con punto para los miles
    if pd.notna(precio):
        precio_texto = f"{precio:,.0f} €".replace(",", ".")
    else:
        precio_texto = "No disponible"

    popup_html = f"""
    <div style="
        font-family: Arial;
        width: 230px;
        line-height: 1.6;
    ">
        <h4>🏠 Vivienda del dataset</h4>
        <b>Superficie:</b> {superficie} m²<br>
        <b>Habitaciones:</b> {habitaciones}<br>
        <b>Baños:</b> {banos}<br>
        <b>Tipo:</b> {tipo}<br>
        <b>Precio:</b> {precio_texto}
    </div>
    """

    folium.CircleMarker(
        location=[
            vivienda.geometry.y,
            vivienda.geometry.x
        ],
        radius=4,
        color="#3388ff",
        fill=True,
        fill_color="#3388ff",
        fill_opacity=0.75,
        weight=1,
        tooltip="🏠 Ver información de la vivienda",
        popup=folium.Popup(
            popup_html,
            max_width=300
        )
    ).add_to(grupo_viviendas)

grupo_viviendas.add_to(mapa)

print("561 viviendas añadidas al mapa.")

A5.35. Incorporación de las capas geoespaciales

In [39]:
def añadir_capa_iconos(
    mapa,
    capa,
    nombre,
    icono,
    color
):
    grupo = folium.FeatureGroup(
        name=nombre
    )

    for _, elemento in capa.iterrows():

        folium.Marker(
            location=[
                elemento.geometry.y,
                elemento.geometry.x
            ],
            icon=folium.Icon(
                icon=icono,
                prefix="fa",
                color=color
            ),
            tooltip=nombre
        ).add_to(grupo)

    grupo.add_to(mapa)


# Añadir las ocho categorías de elementos geoespaciales
añadir_capa_iconos(
    mapa, metro,
    "Estaciones de Metro",
    "subway", "red"
)

añadir_capa_iconos(
    mapa, bus,
    "Paradas de autobús",
    "bus", "blue"
)

añadir_capa_iconos(
    mapa, school,
    "Colegios",
    "graduation-cap", "green"
)

añadir_capa_iconos(
    mapa, hospital,
    "Hospitales",
    "hospital-o", "darkred"
)

añadir_capa_iconos(
    mapa, health_center,
    "Centros de salud",
    "medkit", "orange"
)

añadir_capa_iconos(
    mapa, supermarket,
    "Supermercados",
    "shopping-cart", "purple"
)

añadir_capa_iconos(
    mapa, park,
    "Parques",
    "tree", "green"
)

añadir_capa_iconos(
    mapa, sports,
    "Instalaciones deportivas",
    "futbol-o", "black"
)

print("Las 8 capas geoespaciales se han añadido correctamente.")

Las 8 capas geoespaciales se han añadido correctamente.


A5.36. Control de capas

In [ ]:
folium.LayerControl(
    collapsed=False
).add_to(mapa)

print("Control de capas añadido correctamente.")

A5.37. Captura de la ubicación seleccionada

In [ ]:
from streamlit_folium import st_folium

resultado_mapa = st_folium(
    mapa,
    width=None,
    height=600
)

if resultado_mapa["last_clicked"] is not None:

    latitud = resultado_mapa["last_clicked"]["lat"]
    longitud = resultado_mapa["last_clicked"]["lng"]

    st.session_state.latitud_seleccionada = latitud
    st.session_state.longitud_seleccionada = longitud

    print("Ubicación seleccionada:")
    print("Latitud:", latitud)
    print("Longitud:", longitud)
else:
    print("No se ha seleccionado ninguna ubicación.")

A5.38. Marcador de la ubicación seleccionada

In [ ]:
if (
    st.session_state.latitud_seleccionada is not None
    and st.session_state.longitud_seleccionada is not None
):

    folium.Marker(
        location=[
            st.session_state.latitud_seleccionada,
            st.session_state.longitud_seleccionada
        ],
        icon=folium.Icon(
            icon="home",
            prefix="fa",
            color="green"
        ),
        tooltip="📍 Inmueble a valorar"
    ).add_to(mapa)

    print("Marcador de la vivienda seleccionada añadido.")
else:
    print("No hay ninguna ubicación seleccionada.")

A5.39. Comprobación de la selección y cálculo de distancias

In [ ]:
# Simulación de una ubicación seleccionada
st.session_state.latitud_seleccionada = latitud_prueba
st.session_state.longitud_seleccionada = longitud_prueba

# Recuperación de las coordenadas seleccionadas
latitud_seleccionada = st.session_state.latitud_seleccionada
longitud_seleccionada = st.session_state.longitud_seleccionada

# Cálculo de las distancias
distancias_seleccionada = calcular_distancias_nueva_vivienda(
    latitud_seleccionada,
    longitud_seleccionada,
    capas_visores
)

print("Ubicación seleccionada correctamente:")
print("Latitud:", latitud_seleccionada)
print("Longitud:", longitud_seleccionada)
print()

print("Distancias calculadas:")
for variable, distancia in distancias_seleccionada.items():
    print(f"{variable}: {distancia:.2f} m")

A5.40. Integración del formulario y la valoración

In [44]:
if boton_valorar:

    # Comprobar que existe una ubicación seleccionada
    if (
        st.session_state.latitud_seleccionada is None
        or st.session_state.longitud_seleccionada is None
    ):
        print("Debe seleccionarse una ubicación en el mapa antes de valorar.")

    else:

        # Coordenadas seleccionadas
        latitud = st.session_state.latitud_seleccionada
        longitud = st.session_state.longitud_seleccionada

        # Valoración del inmueble
        precio_estimado, X_nueva = valorar_nueva_vivienda(
            modelo=modelo,
            latitud=latitud,
            longitud=longitud,
            size=size,
            rooms=rooms,
            bathrooms=bathrooms,
            floor=floor,
            hasLift=hasLift,
            status=status,
            propertyType=propertyType,
            typology=typology,
            exterior=exterior,
            newDevelopment=newDevelopment,
            capas_visores=capas_visores
        )

        # Cálculo del valor por metro cuadrado
        precio_m2 = precio_estimado / size

        print("Valoración realizada correctamente.")
        print(
            f"Valor estimado: "
            f"{precio_estimado:,.0f} €".replace(",", ".")
        )
        print(
            f"Valor estimado por m²: "
            f"{precio_m2:,.0f} €/m²".replace(",", ".")
        )

A5.41. Prueba de la valoración integrada

In [ ]:
# Simulación de pulsación del botón de valoración
boton_valorar_prueba = True

if boton_valorar_prueba:

    if (
        st.session_state.latitud_seleccionada is None
        or st.session_state.longitud_seleccionada is None
    ):
        print("Debe seleccionarse una ubicación en el mapa antes de valorar.")

    else:

        latitud = st.session_state.latitud_seleccionada
        longitud = st.session_state.longitud_seleccionada

        precio_estimado, X_nueva = valorar_nueva_vivienda(
            modelo=modelo,
            latitud=latitud,
            longitud=longitud,
            size=size,
            rooms=rooms,
            bathrooms=bathrooms,
            floor=floor,
            hasLift=hasLift,
            status=status,
            propertyType=propertyType,
            typology=typology,
            exterior=exterior,
            newDevelopment=newDevelopment,
            capas_visores=capas_visores
        )

        precio_m2 = precio_estimado / size

        print("Valoración realizada correctamente.")
        print(
            f"Valor estimado: "
            f"{precio_estimado:,.0f} €".replace(",", ".")
        )
        print(
            f"Valor estimado por m²: "
            f"{precio_m2:,.0f} €/m²".replace(",", ".")
        )

A5.42. Presentación del resultado de la valoración

In [ ]:
if boton_valorar_prueba:

    st.success("Valoración realizada correctamente.")

    col1, col2 = st.columns(2)

    with col1:
        st.metric(
            "Valor estimado",
            f"{precio_estimado:,.0f} €".replace(",", ".")
        )

    with col2:
        st.metric(
            "Valor estimado por m²",
            f"{precio_m2:,.0f} €/m²".replace(",", ".")
        )

    st.subheader("Características utilizadas")

    caracteristicas = pd.DataFrame({
        "Característica": [
            "Superficie (m²)",
            "Habitaciones",
            "Baños",
            "Planta",
            "Ascensor",
            "Estado",
            "Tipo de inmueble",
            "Tipología",
            "Exterior",
            "Obra nueva"
        ],
        "Valor": [
            str(size),
            str(rooms),
            str(bathrooms),
            str(floor),
            str(hasLift),
            str(status),
            str(propertyType),
            str(typology),
            str(exterior),
            str(newDevelopment)
        ]
    })

    st.dataframe(
        caracteristicas,
        hide_index=True,
        use_container_width=True
    )

    print("Resultado presentado correctamente en Streamlit.")

A5.43. Presentación de las variables geoespaciales

In [ ]:
if boton_valorar_prueba:

    st.subheader("Variables geoespaciales")

    distancias_mostrar = pd.DataFrame({
        "Elemento": [
            "Metro",
            "Autobús",
            "Colegios",
            "Hospitales",
            "Centros de salud",
            "Supermercados",
            "Parques",
            "Instalaciones deportivas"
        ],
        "Distancia": [
            distancias_seleccionada["dist_metro"],
            distancias_seleccionada["dist_bus"],
            distancias_seleccionada["dist_school"],
            distancias_seleccionada["dist_hospital"],
            distancias_seleccionada["dist_health_center"],
            distancias_seleccionada["dist_supermarket"],
            distancias_seleccionada["dist_park"],
            distancias_seleccionada["dist_sports"]
        ]
    })

    distancias_mostrar["Distancia"] = (
        distancias_mostrar["Distancia"]
        .round(2)
        .astype(str)
        + " m"
    )

    st.dataframe(
        distancias_mostrar,
        hide_index=True,
        width="stretch"
    )

    print("Variables geoespaciales presentadas correctamente.")

A5.44. Resumen de la valoración

In [ ]:
if boton_valorar_prueba:

    st.subheader("Resumen de la valoración")

    st.write(
        f"**Ubicación seleccionada:** "
        f"{latitud_seleccionada:.6f}, "
        f"{longitud_seleccionada:.6f}"
    )

    st.write(
        f"**Superficie del inmueble:** "
        f"{size:.0f} m²"
    )

    st.write(
        f"**Número de habitaciones:** "
        f"{rooms}"
    )

    st.write(
        f"**Número de baños:** "
        f"{bathrooms}"
    )

    st.write(
        f"**Tipo de inmueble:** "
        f"{propertyType}"
    )

    st.write(
        f"**Estado:** "
        f"{status}"
    )

    print("Resumen de la valoración presentado correctamente.")

A5.45. Flujo final de valoración en Streamlit

In [ ]:
# Flujo completo de valoración

if boton_valorar_prueba:

    # Comprobar que existe una ubicación seleccionada
    if (
        st.session_state.latitud_seleccionada is None
        or st.session_state.longitud_seleccionada is None
    ):
        st.warning(
            "Debe seleccionarse una ubicación en el mapa antes de valorar."
        )

    else:

        # Recuperar coordenadas
        latitud = st.session_state.latitud_seleccionada
        longitud = st.session_state.longitud_seleccionada

        # Calcular la valoración
        precio_estimado, X_nueva = valorar_nueva_vivienda(
            modelo=modelo,
            latitud=latitud,
            longitud=longitud,
            size=size,
            rooms=rooms,
            bathrooms=bathrooms,
            floor=floor,
            hasLift=hasLift,
            status=status,
            propertyType=propertyType,
            typology=typology,
            exterior=exterior,
            newDevelopment=newDevelopment,
            capas_visores=capas_visores
        )

        # Calcular valor por metro cuadrado
        precio_m2 = precio_estimado / size

        # Presentar resultados
        st.success("Valoración realizada correctamente.")

        col1, col2 = st.columns(2)

        with col1:
            st.metric(
                "Valor estimado",
                f"{precio_estimado:,.0f} €".replace(",", ".")
            )

        with col2:
            st.metric(
                "Valor estimado por m²",
                f"{precio_m2:,.0f} €/m²".replace(",", ".")
            )

        print("Flujo completo de valoración ejecutado correctamente.")
        print(
            f"Valor estimado: "
            f"{precio_estimado:,.0f} €".replace(",", ".")
        )
        print(
            f"Valor estimado por m²: "
            f"{precio_m2:,.0f} €/m²".replace(",", ".")
        )

A5.46. Flujo final de la aplicación

In [ ]:
# Comprobación de ubicación seleccionada
if (
    st.session_state.latitud_seleccionada is not None
    and st.session_state.longitud_seleccionada is not None
):

    latitud = st.session_state.latitud_seleccionada
    longitud = st.session_state.longitud_seleccionada

    if boton_valorar:

        precio_estimado, X_nueva = valorar_nueva_vivienda(
            modelo=modelo,
            latitud=latitud,
            longitud=longitud,
            size=size,
            rooms=rooms,
            bathrooms=bathrooms,
            floor=floor,
            hasLift=hasLift,
            status=status,
            propertyType=propertyType,
            typology=typology,
            exterior=exterior,
            newDevelopment=newDevelopment,
            capas_visores=capas_visores
        )

        precio_m2 = precio_estimado / size

        st.success("Valoración realizada correctamente.")

        col1, col2 = st.columns(2)

        with col1:
            st.metric(
                "Valor estimado",
                f"{precio_estimado:,.0f} €".replace(",", ".")
            )

        with col2:
            st.metric(
                "Valor estimado por m²",
                f"{precio_m2:,.0f} €/m²".replace(",", ".")
            )

else:

    st.info(
        "Seleccione una ubicación en el mapa para realizar una valoración."
    )

print("Flujo final de la aplicación definido correctamente.")

A5.47. Preparación del resultado final

In [ ]:
# Preparación del resultado final de la valoración

if (
    st.session_state.latitud_seleccionada is not None
    and st.session_state.longitud_seleccionada is not None
):

    latitud = st.session_state.latitud_seleccionada
    longitud = st.session_state.longitud_seleccionada

    print("Ubicación seleccionada:")
    print(f"Latitud: {latitud:.6f}")
    print(f"Longitud: {longitud:.6f}")
    print()

    print("La aplicación está preparada para realizar la valoración.")

else:

    print(
        "No se ha seleccionado ninguna ubicación."
    )

A5.48. Cálculo de las variables geoespaciales de la ubicación seleccionada

In [ ]:
if (
    st.session_state.latitud_seleccionada is not None
    and st.session_state.longitud_seleccionada is not None
):

    latitud = st.session_state.latitud_seleccionada
    longitud = st.session_state.longitud_seleccionada

    distancias_finales = calcular_distancias_nueva_vivienda(
        latitud,
        longitud,
        capas_visores
    )

    print("Variables geoespaciales calculadas correctamente.")
    print()

    for variable, distancia in distancias_finales.items():
        print(
            f"{variable}: "
            f"{distancia:.2f} m"
        )

else:

    print(
        "No se puede calcular las distancias "
        "sin una ubicación seleccionada."
    )

A5.49. Construcción de la observación final

In [ ]:
if (
    st.session_state.latitud_seleccionada is not None
    and st.session_state.longitud_seleccionada is not None
):

    latitud = st.session_state.latitud_seleccionada
    longitud = st.session_state.longitud_seleccionada

    X_final = pd.DataFrame([{
        "size": size,
        "rooms": rooms,
        "bathrooms": bathrooms,
        "floor": floor,
        "hasLift": hasLift,
        "status": status,
        "propertyType": propertyType,
        "typology": typology,
        "exterior": exterior,
        "newDevelopment": newDevelopment,
        "latitude": latitud,
        "longitude": longitud,
        "dist_metro": distancias_finales["dist_metro"],
        "dist_bus": distancias_finales["dist_bus"],
        "dist_school": distancias_finales["dist_school"],
        "dist_hospital": distancias_finales["dist_hospital"],
        "dist_health_center": distancias_finales["dist_health_center"],
        "dist_supermarket": distancias_finales["dist_supermarket"],
        "dist_park": distancias_finales["dist_park"],
        "dist_sports": distancias_finales["dist_sports"]
    }])

    print("Observación final construida correctamente.")
    print("Número de variables:", X_final.shape[1])
    print()
    print(X_final)

else:

    print(
        "No se puede construir la observación "
        "sin una ubicación seleccionada."
    )

A5.50. Predicción final del modelo

In [ ]:
if "X_final" in globals():

    precio_final = modelo.predict(
        X_final
    )[0]

    precio_m2_final = precio_final / size

    print("Predicción final realizada correctamente.")
    print(
        f"Valor estimado: "
        f"{precio_final:,.0f} €".replace(",", ".")
    )
    print(
        f"Valor estimado por m²: "
        f"{precio_m2_final:,.0f} €/m²".replace(",", ".")
    )

else:

    print(
        "No se puede realizar la predicción "
        "porque no existe la observación final."
    )

A5.51. Presentación final de la valoración

In [ ]:
if "precio_final" in globals():

    print("========================================")
    print("       VALORACIÓN DEL INMUEBLE")
    print("========================================")
    print()
    print(
        f"Valor estimado: "
        f"{precio_final:,.0f} €".replace(",", ".")
    )
    print(
        f"Valor estimado por m²: "
        f"{precio_m2_final:,.0f} €/m²".replace(",", ".")
    )
    print()
    print("Características del inmueble:")
    print(f"- Superficie: {size:.0f} m²")
    print(f"- Habitaciones: {rooms}")
    print(f"- Baños: {bathrooms}")
    print(f"- Planta: {floor}")
    print(f"- Ascensor: {hasLift}")
    print(f"- Estado: {status}")
    print(f"- Tipo de inmueble: {propertyType}")
    print(f"- Tipología: {typology}")
    print(f"- Exterior: {exterior}")
    print(f"- Obra nueva: {newDevelopment}")
    print()
    print("Variables geoespaciales:")

    for variable, distancia in distancias_finales.items():
        print(
            f"- {variable}: "
            f"{distancia:.2f} m"
        )

    print()
    print("Valoración final presentada correctamente.")

else:

    print(
        "No existe una predicción final."
    )

In [79]:
!pip install -q streamlit streamlit-folium

In [ ]:
!pkill -f "streamlit run app.py" || true

In [81]:
!streamlit run app.py --server.headless true --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 &

In [ ]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print(url)